# Stage 11, model v2 set — the one v2 FoR held-out run (D48 (7))

This notebook carries out the separate `stage11_test` claim for the model v2 set (D48 (7); Protocol §7 addendum; D31, D41, D45, D46, D57, D58, D64) end to end on a Colab T4 runtime, from an exact clean commit, in notebook 06's proven order, with the runner `src.evaluation.stage11_v2_run` ([docs/stage11_run.md](../docs/stage11_run.md), "v2 model set" and "Notebook 09") and the report `src.evaluation.stage11_v2_report` ([docs/stage11_report.md](../docs/stage11_report.md), "v2 model set"). The model set is the three seeds of the v2 search winner; the winner, its input kind and the bundle seed are read in code from the committed search evidence, never typed. There is no baseline (it was scored in the v1 claim), so no MFCC and no D58 reference. It has two modes:

- `MODE = 'dry_run'` (committed default): environment gate (with the D64 version guard), winner from evidence, pytest, reproduction-gate caches (log-Mel winners only), artifacts, the read-only registry pre-check, then the **rehearsal** on val-A and val-B (known answers), its evidence exported to Drive, then it **stops**. No claim, no held-out access. Rerunnable: each dry run keeps its own timestamped folder.
- `MODE = 'official'`: the same steps, then the registry **claim** → Drive flush and remount, claim verified on Drive → **score** → **report** → export → only then the results view. Runs once: the claim cannot be repeated (D46 (8)).

The notebook never names the held-out split: the runner fixes it in code (there is no split argument), and the guard `tests/governance/test_for_test_access.py` scans this file. Nothing is logged to an external service; the committed evidence is the record (D40). Run it only after the search evidence PR (`docs/audit_evidence/v2_search/`) and the risk-v2 results PR (`docs/audit_evidence/v2_risk/`) are merged, from a reviewed `main` commit that holds both. F5 reads this run's committed evidence, so F5 is built only after this notebook's results PR is merged.

## 1. Parameters

Before running, select the Colab runtime **T4 GPU** and, under "Version d'exécution / Runtime version", **2026.10** (D58 (1): Python 3.13.16 and torch 2.11.0+cu130); the environment cell asserts both versions exactly. Replace `COMMIT` with the full 40-character approved commit. `MODE` is `'dry_run'` or `'official'`. `RISK_VERSION` names the risk-v2 folder on Drive the stored val-B scores are copied from and must equal the `version` in the committed risk-v2 run report. `SEARCH_VERSION` is notebook 08's constant: it names the search folder on Drive the checkpoints are copied from and must equal the `version` in the committed search report. `RESULTS_REF` is the repository path where the results PR will commit `metrics_report.md`; `claim` stores it and `complete` records it in the registry. There is no winner, seed or path knob: the winner and the bundle seed come from the committed search evidence, the batch size from `configs/v2_candidates.yaml`, everything else from the committed evidence.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'  # full 40-character SHA
MODE = 'dry_run'  # 'dry_run' or 'official'
RISK_VERSION = 'risk-v2'
SEARCH_VERSION = 'model-v2'  # notebook 08's constant: the search folder and the search reports' version
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'
RESULTS_REF = 'docs/results/stage11_v2_for_test.md'
assert MODE in ('dry_run', 'official'), MODE

## 2. Mount Drive and refuse existing outputs

All paths are constants under `DRIVE_ROOT`, as in the "Drive layout (Track V)" table of `docs/v2_training.md`; nothing is typed by the operator:

| Constant | Drive path | Here |
|---|---|---|
| `REGISTRY` | `processed/metadata/stage11/run_registry.jsonl` | the live run registry shared with v1 (D46 (6), D48 (7)): the v2 claim appends to it; written only by the runner's `claim` and `complete`, never deleted, moved or rewritten by this notebook |
| `RUN_DEST` | `processed/metadata/stage11_v2/run/` | official mode: `runner/` (`CLAIM_OUT`, written in place by `claim` and `score`: claim and score evidence, `features.json`, `clip_scores/`), the metrics report, `rehearsal/`, the export manifest |
| `DRY_RUN_DEST` | `processed/metadata/stage11_v2/dry_run/<UTC timestamp>/` | dry run: `rehearsal/` (written in place by `rehearse`) and the manifest, one folder per dry run |
| `SEARCH_OUT` | `checkpoints/v2_search/<SEARCH_VERSION>/` (`runs/<winner>_seed<k>/checkpoint/`) | read (the winner's three seeds only) |
| `SELECTION_SET_DRIVE` | `processed/v2/selection_set_s/` | read |
| `RISK_DRIVE` | `checkpoints/risk/<RISK_VERSION>/` (`val_b_scores.npz`) | read |
| FoR archive | `raw/for_2sec_ff8c82c7.tar` and its revision marker | read |

The v2 outputs go to the new folder `stage11_v2/`; v1's `processed/metadata/stage11/run/` and `dry_run/` are never touched (only the shared registry next to them is appended to, by the runner). Official mode stops here if `run/` exists; a dry run stops if its timestamped folder exists. The same checks run again right before the rehearsal and the claim.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
REGISTRY = DRIVE / 'processed' / 'metadata' / 'stage11' / 'run_registry.jsonl'
STAGE11_V2 = DRIVE / 'processed' / 'metadata' / 'stage11_v2'
RUN_DEST = STAGE11_V2 / 'run'
DRY_RUN_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRY_RUN_DEST = STAGE11_V2 / 'dry_run' / DRY_RUN_STAMP
SEARCH_OUT = DRIVE / 'checkpoints' / 'v2_search' / SEARCH_VERSION
SELECTION_SET_DRIVE = DRIVE / 'processed' / 'v2' / 'selection_set_s'
RISK_DRIVE = DRIVE / 'checkpoints' / 'risk' / RISK_VERSION


def refuse_existing_outputs():
    dest = RUN_DEST if MODE == 'official' else DRY_RUN_DEST
    if dest.exists():
        raise FileExistsError(f'{dest} already exists; v2 Stage 11 outputs are never overwritten')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


refuse_existing_outputs()
assert (DRIVE / 'raw').is_dir(), f'{DRIVE / "raw"} not found; check DRIVE_ROOT'
print('MODE', MODE)

## 3. Check out the exact commit

As notebook 06: the repository is cloned into `/content` and the full commit is checked out detached. HEAD must equal `COMMIT` exactly and the worktree must be clean. `assert_clean` also refuses ignored files other than Python bytecode caches, because `.gitignore` hides some folders from `git status`; it runs before and after every long cell (install, winner probe, archive, pytest, caches, copies, each runner call, report). Every later output (caches, pytest XML, artifact copies, runner and report output) is written outside the clone, so the runner, which refuses a dirty worktree, sees a clean tree. `stream` prints a child's output as it runs; `probe` runs a short script in a fresh process inside the clone and returns its JSON output.

In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout


def assert_clean():
    head = git('rev-parse', 'HEAD').strip()
    assert head == COMMIT, (head, COMMIT)
    status = git('status', '--porcelain')
    assert not status, status
    ignored = [line[3:] for line in git('status', '--porcelain', '--ignored').splitlines() if line.startswith('!! ')]
    unexpected = [path for path in ignored if not path.rstrip('/').endswith('__pycache__')]
    assert not unexpected, unexpected


def stream(command, **kwargs):
    # Like subprocess.run(check=True), but prints the child's output line by line as it runs.
    env = {**kwargs.pop('env', os.environ), 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                          **kwargs) as process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def probe(script):
    # Run a short Python script in a fresh process inside the clone; return its JSON output.
    result = subprocess.run([sys.executable, '-c', script], cwd=REPO, capture_output=True, text=True)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
    return json.loads(result.stdout)


assert_clean()
os.chdir(REPO)
print('HEAD', COMMIT)

## 4. Environment gate (D46 (1), (2)) and the D64 version guard

The runtime must be a Colab **T4** runtime (D46 (1): the runtime that reproduces the Stage 8 val-A log-Mel bytes, Entry 94). The GPU is **never used for scoring**: the runner scores every model on CPU, because risk-v2 was fitted on CPU scores, and no runner call takes a device argument. The GPU name is only asserted here and recorded in the export manifest.

The Colab runtime version must be **2026.10** ("Version d'exécution / Runtime version", selected with the T4 before running; D58 (1)): Python must be exactly 3.13.16 (D15, D58 (1)) and Colab's preinstalled torch exactly 2.11.0+cu130 (`requirements-cnn.txt`, D42, D58 (1)); any other image stops here. Torch is never installed by this notebook: only `requirements.txt` and `requirements-cnn.txt` are installed, and neither lists torch.

Then, in a **fresh process** after the install, Python and torch are read again (a silent upgrade during the install is caught) and `src.augmentation.version_guard.require_versions()` runs: NumPy, SciPy and SoundFile must equal the `requirements.txt` pins (D64: an unpinned SciPy once changed numeric output). Its result is printed and recorded in the export manifest; any mismatch stops the notebook here. This is the same guard notebooks 08 and 10 call; the notebook has no version helper of its own. The runner asserts Python and torch again and records the full D46 (3) environment itself.

In [ ]:
import torch

GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          check=True, capture_output=True, text=True).stdout.strip()
assert 'T4' in GPU_NAME, f'D46 (1) requires a Colab T4 runtime, got {GPU_NAME!r}'
assert torch.__version__ == '2.11.0+cu130', f'D58 (1) requires Colab runtime 2026.10 (torch 2.11.0+cu130), got {torch.__version__}'
assert_clean()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-cnn.txt'],
               cwd=REPO, check=True)
ENV = probe('''
import json, platform, numpy, torch
from src.augmentation.version_guard import require_versions
print(json.dumps({
    'python': platform.python_version(),
    'numpy': numpy.__version__,
    'torch': torch.__version__,
    'require_versions': require_versions(),
}))
''')
assert ENV['torch'] == torch.__version__, (ENV['torch'], torch.__version__)
assert ENV['torch'] == '2.11.0+cu130', f"D58 (1) requires torch 2.11.0+cu130 (runtime 2026.10), got {ENV['torch']}"
assert ENV['python'] == '3.13.16', f"D58 (1) requires Python 3.13.16 (runtime 2026.10), got {ENV['python']}"
VERSIONS = ENV.pop('require_versions')
assert_clean()
print('require_versions:', VERSIONS)
for key, value in {**ENV, 'gpu_name': GPU_NAME}.items():
    print(f'{key}: {value}')

## 5. The winner and its bundle seed, from the committed evidence

The winner, the bundle seed and the bundle checkpoint's SHA-256 are read with the runner's own rule, `v2_risk_run.bundle_checkpoint(docs/audit_evidence/v2_search)`, in a fresh process (the two search reports must agree; no rule is duplicated here), together with the winner's input kind (`src.models.v2_checkpoint.INPUTS`) and the three run folder names (`v2_search.run_dir_name` for the runner's `SEEDS`). The committed search `run_report.json` must name `SEARCH_VERSION`, so the checkpoints are taken from the search folder the evidence describes; the committed risk-v2 `run_report.json` (`docs/audit_evidence/v2_risk/`, the file whose `version` field binds the policy version) must name `RISK_VERSION`, so the stored val-B scores are taken from the risk folder the evidence describes. Both are printed. The F3 profile file must match the Stage 6 evidence. The runner re-reads and cross-checks all of this evidence itself before any check.

In [ ]:
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
F3 = json.loads((REPO / 'docs/audit_evidence/stage6_preprocessing/profile_report.json').read_text(encoding='utf-8'))['profile']
assert sha256(PROFILE) == F3['config_sha256'], 'configs/preprocessing.yaml differs from F3'
SEARCH_EVIDENCE = REPO / 'docs/audit_evidence/v2_search'
RISK_EVIDENCE = REPO / 'docs/audit_evidence/v2_risk'
assert (SEARCH_EVIDENCE / 'run_report.json').is_file(), f'{SEARCH_EVIDENCE} is missing: merge the search evidence PR first'
assert (RISK_EVIDENCE / 'run_report.json').is_file(), f'{RISK_EVIDENCE} is missing: merge the risk-v2 results PR first'
SEARCH_REPORT = json.loads((SEARCH_EVIDENCE / 'run_report.json').read_text(encoding='utf-8'))
assert SEARCH_REPORT['version'] == SEARCH_VERSION, (SEARCH_REPORT['version'], SEARCH_VERSION)
RISK_REPORT = json.loads((RISK_EVIDENCE / 'run_report.json').read_text(encoding='utf-8'))
assert RISK_REPORT['version'] == RISK_VERSION, (RISK_REPORT['version'], RISK_VERSION)
assert_clean()
BUNDLE = probe(f'''
import json
from pathlib import Path
from src.evaluation.stage11_v2_run import SEEDS
from src.models.v2_checkpoint import INPUTS
from src.tuning.v2_risk_run import bundle_checkpoint
from src.tuning.v2_search import run_dir_name
winner, seed, checkpoint_sha256 = bundle_checkpoint(Path({str(SEARCH_EVIDENCE)!r}))
print(json.dumps({{'winner': winner, 'bundle_seed': seed, 'checkpoint_sha256': checkpoint_sha256,
                  'input_kind': INPUTS[winner], 'run_dirs': [run_dir_name(winner, k) for k in SEEDS]}}))
''')
WINNER, BUNDLE_SEED = BUNDLE['winner'], BUNDLE['bundle_seed']
INPUT_KIND, RUN_DIRS = BUNDLE['input_kind'], BUNDLE['run_dirs']
assert INPUT_KIND in ('logmel', 'waveform'), INPUT_KIND
assert len(set(RUN_DIRS)) == len(RUN_DIRS) == 3, RUN_DIRS
assert_clean()
print(json.dumps(BUNDLE, indent=2))
print('search version:', SEARCH_REPORT['version'], '| risk version:', RISK_REPORT['version'])

## 6. Verify and extract the pinned FoR archive

As notebook 06: the revision marker next to the Drive backup must equal the D21 FoR revision, read here from `configs/audit.yaml` (the D21 pin location). The archive is extracted with `filter='data'`. Its SHA-256 is computed and written to the export manifest as `for_archive_sha256_observed`: this is evidence only, never asserted or compared, and not a dataset pin (D21 unchanged).

In [ ]:
import tarfile
import yaml

EXPECTED_REVISION = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']['revision']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION_FILE = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / 'for_2sec_ff8c82c7.tar'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert_clean()
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
FOR_ARCHIVE_SHA256_OBSERVED = sha256(LOCAL_ARCHIVE)  # recorded only (evidence, not a pin)
assert_clean()
print('FoR archive extracted to', AUDIO_ROOT)
print('for_archive_sha256_observed:', FOR_ARCHIVE_SHA256_OBSERVED)

## 7. Run the test suite: no failure, error or skip (D46 (2))

`REQUIRE_TORCH_TESTS=1` turns a missing torch into a failure. The counts come from the JUnit XML written to `/content` (outside the clone), not from the terminal text, and every failure, error and skip stops the notebook.

In [ ]:
import xml.etree.ElementTree as ET

JUNIT = Path('/content/pytest_stage11_v2.xml')
if JUNIT.exists():
    JUNIT.unlink()
assert_clean()
stream([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', f'--junitxml={JUNIT}'],
       cwd=REPO, env={**os.environ, 'REQUIRE_TORCH_TESTS': '1'})
root = ET.parse(JUNIT).getroot()
suites = [root] if root.tag == 'testsuite' else root.findall('testsuite')
COUNTS = {key: sum(int(suite.get(key, 0)) for suite in suites) for key in ('tests', 'failures', 'errors', 'skipped')}
print(COUNTS)
assert COUNTS['tests'] > 0, COUNTS
assert COUNTS['failures'] == COUNTS['errors'] == COUNTS['skipped'] == 0, COUNTS
assert_clean()

## 8. Reproduction-gate caches (D46 (4)), log-Mel winners only

**`cnn-aug` and `crnn` only** (`INPUT_KIND == 'logmel'`). The runner reads two log-Mel cache builds in the layout `<cache>/log_mel/<split>/` (`--cache-a`, `--cache-b`; `src/evaluation/stage11_v2_run.py`). Build A, `/content/cache_a`, holds val-A and val-B (val-B is needed by the rehearsal: the cache risk-v2 was fitted on); build B, `/content/cache_b`, holds val-A only (the gate rebuilds val-A only). Every sample hash is checked before decoding. Here the val-A files of both builds must be byte-identical, and every cache must record the F3 profile config SHA-256 and fingerprint stored in the repo (Stage 6 evidence, D39) and `COMMIT`. The runner then re-checks the val-A `npz_sha256` against Stage 8 and the val-B one against risk-v2's `val_b_input`. No MFCC cache and no D58 reference: the v2 model set has no baseline.

**`aasist`** (`INPUT_KIND == 'waveform'`): no cache is built and no `--cache-*` argument is passed (the runner refuses them); its gate compares the in-memory val-A F3 waveforms (two passes) with S's clean waveforms.

Only val-A and val-B are ever built; no cache is built for the training or held-out splits.

In [ ]:
CACHE_A, CACHE_B = Path('/content/cache_a'), Path('/content/cache_b')
CACHE_HASHES_A = CACHE_HASHES_B = None


def file_hashes(folder):
    return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(folder.rglob('*')) if p.is_file()}


if INPUT_KIND == 'logmel':
    CACHE_PLAN = ((CACHE_A, ('val-A', 'val-B')), (CACHE_B, ('val-A',)))
    for cache, splits in CACHE_PLAN:
        if cache.exists():
            shutil.rmtree(cache)
        cache.mkdir()
        for split in splits:
            assert_clean()
            subprocess.run([
                sys.executable, '-m', 'src.features.logmel_cache',
                '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE),
                '--split', split, '--out', str(cache / 'log_mel' / split),
            ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)
            assert_clean()
    CACHE_HASHES_A, CACHE_HASHES_B = file_hashes(CACHE_A), file_hashes(CACHE_B)
    for cache, splits in CACHE_PLAN:
        built = sorted(file_hashes(cache))
        planned = sorted(f'log_mel/{s}/{s}.{ext}' for s in splits for ext in ('npz', 'json'))
        assert built == planned, (cache, built)
    CACHE_A_VAL_A = {rel: digest for rel, digest in CACHE_HASHES_A.items() if rel.split('/')[1] == 'val-A'}
    assert CACHE_HASHES_B == CACHE_A_VAL_A, 'the two val-A cache builds differ'
    for cache, splits in CACHE_PLAN:
        for split in splits:
            meta = json.loads((cache / 'log_mel' / split / f'{split}.json').read_text(encoding='utf-8'))
            assert meta['profile_config_sha256'] == F3['config_sha256'], (cache, split)
            assert meta['profile_fingerprint'] == F3['fingerprint'], (cache, split)
            assert meta['code_commit'] == COMMIT, (cache, split)
    print(json.dumps(CACHE_HASHES_A, indent=2))
else:
    print(f'{WINNER} takes the F3 waveform: no cache is built and no --cache-* argument is passed')
assert_clean()

## 9. Copy the artifacts from Drive

Copied to `/content/stage11_v2_artifacts/`, outside the clone, and never loaded here: the winner's three checkpoint folders `checkpoints/v2_search/<SEARCH_VERSION>/runs/<winner>_seed<k>/checkpoint/` for k = 0, 1, 2 (only these three: `runs/` locally holds exactly them, the layout `--checkpoints` expects), the selection set S (`processed/v2/selection_set_s/`) and risk-v2's stored val-B scores `checkpoints/risk/<RISK_VERSION>/val_b_scores.npz` (passed to `rehearse` only). The risk policy is not copied: the runner reads `risk_policy.json` from the committed `docs/audit_evidence/v2_risk/`. Nothing is verified here: the runner checks every checkpoint hash against the committed search evidence, S against the hash the checkpoints and the search recorded, and the val-B scores against risk-v2's `val_b_scores_sha256`, before it uses any of them.

In [ ]:
ARTIFACTS = Path('/content/stage11_v2_artifacts')
CHECKPOINTS_LOCAL = ARTIFACTS / 'runs'
S_LOCAL = ARTIFACTS / 'selection_set_s'
RISK_LOCAL = ARTIFACTS / 'risk' / RISK_VERSION
assert_clean()
if ARTIFACTS.exists():
    shutil.rmtree(ARTIFACTS)
for run_dir in RUN_DIRS:  # the winner's three seeds, nothing else from runs/
    source = SEARCH_OUT / 'runs' / run_dir / 'checkpoint'
    assert source.is_dir(), f'{source} not found'
    shutil.copytree(source, CHECKPOINTS_LOCAL / run_dir / 'checkpoint')
assert SELECTION_SET_DRIVE.is_dir(), f'{SELECTION_SET_DRIVE} not found'
shutil.copytree(SELECTION_SET_DRIVE, S_LOCAL)
RISK_LOCAL.mkdir(parents=True)
shutil.copy2(RISK_DRIVE / 'val_b_scores.npz', RISK_LOCAL / 'val_b_scores.npz')
assert sorted(p.name for p in CHECKPOINTS_LOCAL.iterdir()) == sorted(RUN_DIRS)
assert not ARTIFACTS.resolve().is_relative_to(REPO.resolve()), ARTIFACTS
assert_clean()
for path in sorted(ARTIFACTS.rglob('*')):
    if path.is_file() and not path.is_relative_to(S_LOCAL):
        print(path.relative_to(ARTIFACTS).as_posix())
print('S:', sum(1 for p in S_LOCAL.rglob('*') if p.is_file()), 'files')

## 10. Registry pre-check (read-only)

Before the rehearsal, and so before any claim, the live Drive registry must be byte-identical to the reviewed committed copy `docs/audit_evidence/run_registry.jsonl` (the v1 pair, D46 (6)): same SHA-256 and same line count. A mismatch (an unreviewed record, a lost line, a stale Drive copy) stops here, before the claim, with both hashes and line counts, for team review. The registry is only read here, never written; the runner's own registry rule (the v1 pair and no v2 record, D48 (7)) runs again inside `claim`. The result is recorded in the manifests as `registry_pre_check`, and the claim cell re-checks the Drive hash right before the claim.

In [ ]:
COMMITTED_REGISTRY = REPO / 'docs/audit_evidence/run_registry.jsonl'
REGISTRY_PRE_CHECK = {
    'drive': {'sha256': sha256(REGISTRY), 'lines': len(REGISTRY.read_bytes().splitlines())},
    'committed': {'sha256': sha256(COMMITTED_REGISTRY), 'lines': len(COMMITTED_REGISTRY.read_bytes().splitlines())},
}
if REGISTRY_PRE_CHECK['drive'] != REGISTRY_PRE_CHECK['committed']:
    raise ValueError(f'the Drive registry is not the reviewed committed one; stop before the claim: {REGISTRY_PRE_CHECK}')
print('Registry pre-check passed:', json.dumps(REGISTRY_PRE_CHECK['drive']))

## 11. Rehearsal (the dry run, known answers)

`rehearse` runs the same pre-claim checks as the official run (environment gate, evidence, checkpoint, S, policy and batch-size checks, the reproduction gate), then checks known answers on val-A and val-B only: per seed, the five S EERs and the selection metric M recomputed on CPU must equal the search's `best_selection_m` exactly; on val-B, the in-memory input must equal the one risk-v2 was fitted on, and the bundle seed's raw scores and `p` must equal risk-v2's `val_b_scores.npz` byte for byte ([docs/stage11_run.md](../docs/stage11_run.md), "v2 model set", "Rehearsal"). It opens no registry and reads only val-A and val-B. `INPUTS` holds the arguments shared by `rehearse`, `claim` and `score`; `--cache-a` / `--cache-b` are added for a log-Mel winner only. It writes `rehearsal_report.json` and `pip_freeze.txt` into a new folder: in dry-run mode `dry_run/<UTC timestamp>/rehearsal/` on Drive (the folder `dry_run/<UTC timestamp>/` is created here with `exist_ok=False`), so the dry run also proves that the runner's folder creation and exclusive writes work on Drive, as the official run needs; in official mode the local `/content/stage11_v2_rehearsal`, copied to `run/rehearsal/` at export.

If the runner raises, the notebook stops here; the runner has already written the failed report (`"passed": false`, `failed_check`, the error) into that folder for the team. A rehearsal that ends with a failed check (`failed_check` not null) stops the v2 Stage 11 run until a logged decision; one that crashes on an infrastructure error (`failed_check` null) may be rerun from the same COMMIT (D46 (8)). Only `passed` is read from the report.

In [ ]:
RUNNER = 'src.evaluation.stage11_v2_run'
if INPUT_KIND == 'logmel':
    CACHE_ARGS = ['--cache-a', str(CACHE_A), '--cache-b', str(CACHE_B)]
else:
    CACHE_ARGS = []  # aasist: the runner refuses cache arguments
INPUTS = [
    '--checkpoints', str(CHECKPOINTS_LOCAL), '--selection-set', str(S_LOCAL), '--profile', str(PROFILE),
    '--config', str(REPO / 'configs/v2_candidates.yaml'), '--manifest', str(MANIFEST), *CACHE_ARGS,
]
if MODE == 'dry_run':
    refuse_existing_outputs()
    DRY_RUN_DEST.mkdir(parents=True, exist_ok=False)
    REHEARSAL_OUT = DRY_RUN_DEST / 'rehearsal'  # on Drive; the runner creates it
else:
    REHEARSAL_OUT = Path('/content/stage11_v2_rehearsal')
if REHEARSAL_OUT.exists():
    raise FileExistsError(f'{REHEARSAL_OUT} already exists; the rehearsal writes a new folder')
assert_clean()
stream([sys.executable, '-m', RUNNER, 'rehearse', '--root', str(AUDIO_ROOT), *INPUTS,
        '--risk-scores', str(RISK_LOCAL / 'val_b_scores.npz'), '--out', str(REHEARSAL_OUT)], cwd=REPO)
assert_clean()
REHEARSAL_PASSED = json.loads((REHEARSAL_OUT / 'rehearsal_report.json').read_text(encoding='utf-8'))['passed']
assert REHEARSAL_PASSED is True, 'the rehearsal did not pass; the v2 Stage 11 run stops (D46 (8))'
assert sorted(p.name for p in REHEARSAL_OUT.iterdir()) == ['pip_freeze.txt', 'rehearsal_report.json']
print('Rehearsal passed.')

## 12. Dry-run export, then stop

**Dry-run mode only.** The rehearsal already wrote `rehearsal_report.json` and `pip_freeze.txt` to `processed/metadata/stage11_v2/dry_run/<UTC timestamp>/rehearsal/` on Drive; they are hashed there, in place, and `stage11_v2_dry_run_manifest.json` is written next to `rehearsal/`. The manifest records the commit, the mode, `SEARCH_VERSION`, `RISK_VERSION`, the winner, its input kind and the bundle seed, the environment (Python, NumPy, torch, GPU name) and the `require_versions` result, the pytest counts, the SHA-256 of every cache file of both builds (`null` for `aasist`), the F3 values, the D21 FoR revision and the observed archive SHA-256 (evidence only), the registry pre-check, the rehearsal's `passed`, and each exported file's SHA-256. Drive is then flushed, unmounted and remounted, and every file is hashed again against the manifest (a missing, extra or changed file fails).

In official mode this cell only defines the export helpers: the rehearsal evidence is exported later with the run, under `run/rehearsal/` (step 17). A dry run ends here; the next cell refuses to run in dry-run mode.

In [ ]:
def drive_entry(path):
    # Manifest entry of a file already on Drive, hashed in place.
    return {'path': path.relative_to(DRIVE).as_posix(), 'size': path.stat().st_size, 'sha256': sha256(path)}


def export_file(source, dest):
    # Copy one file to Drive, refusing to overwrite, and return its manifest entry.
    assert not dest.exists(), f'{dest} already exists'
    shutil.copy2(source, dest)
    entry = drive_entry(dest)
    assert entry['sha256'] == sha256(source), f'{dest} differs from {source}'
    return entry


def verify_after_remount(root, manifest):
    # Every file under root on Drive must match the manifest: no missing, extra or changed file.
    expected = {entry['path']: entry['sha256'] for entry in manifest['files']}
    stored = {p.relative_to(DRIVE).as_posix() for p in root.rglob('*') if p.is_file()}
    missing, extra = sorted(set(expected) - stored), sorted(stored - set(expected))
    assert not missing and not extra, f'missing on Drive: {missing}; extra on Drive: {extra}'
    mismatched = [path for path, digest in sorted(expected.items()) if sha256(DRIVE / path) != digest]
    assert not mismatched, f'SHA-256 mismatch on Drive after remount: {mismatched}'
    return len(expected)


def base_manifest():
    return {
        'commit': COMMIT,
        'mode': MODE,
        'search_version': SEARCH_VERSION,
        'risk_version': RISK_VERSION,
        'winner': WINNER,
        'input_kind': INPUT_KIND,
        'bundle_seed': BUNDLE_SEED,
        'python': ENV['python'],
        'numpy': ENV['numpy'],
        'torch': ENV['torch'],
        'require_versions': VERSIONS,
        'gpu_name': GPU_NAME,
        'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
        'for_revision': EXPECTED_REVISION,
        'for_archive_sha256_observed': FOR_ARCHIVE_SHA256_OBSERVED,
        'pytest_counts': COUNTS,
        'cache_sha256': {'cache_a': CACHE_HASHES_A, 'cache_b': CACHE_HASHES_B},
        'f3': {'config_sha256': F3['config_sha256'], 'fingerprint': F3['fingerprint']},
        'registry_pre_check': REGISTRY_PRE_CHECK,
        'rehearsal_passed': REHEARSAL_PASSED,
    }


if MODE == 'dry_run':
    assert REHEARSAL_OUT == DRY_RUN_DEST / 'rehearsal', REHEARSAL_OUT
    stored = sorted(p.relative_to(DRY_RUN_DEST).as_posix() for p in DRY_RUN_DEST.rglob('*') if p.is_file())
    assert stored == ['rehearsal/pip_freeze.txt', 'rehearsal/rehearsal_report.json'], stored
    files = [drive_entry(DRY_RUN_DEST / rel) for rel in stored]
    DRY_RUN_MANIFEST = {**base_manifest(), 'files': files}
    DRY_RUN_MANIFEST_PATH = DRY_RUN_DEST / 'stage11_v2_dry_run_manifest.json'
    assert not DRY_RUN_MANIFEST_PATH.exists(), DRY_RUN_MANIFEST_PATH
    DRY_RUN_MANIFEST_TEXT = json.dumps(DRY_RUN_MANIFEST, indent=2, sort_keys=True) + '\n'
    DRY_RUN_MANIFEST_PATH.write_text(DRY_RUN_MANIFEST_TEXT, encoding='utf-8')
    print(DRY_RUN_MANIFEST_TEXT, end='')
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    checked = {'files': [*DRY_RUN_MANIFEST['files'], {
        'path': DRY_RUN_MANIFEST_PATH.relative_to(DRIVE).as_posix(),
        'sha256': hashlib.sha256(DRY_RUN_MANIFEST_TEXT.encode('utf-8')).hexdigest()}]}
    print(f'Drive verified after remount: {verify_after_remount(DRY_RUN_DEST, checked)} files match the manifest.')
    print('Dry run complete. Stop here.')

## 13. Claim (D46 (5), D48 (7)) — official mode only

From here on every code cell starts with `assert MODE == 'official'`, so a dry run cannot go further.

`refuse_existing_outputs()` checks right before the claim that `run/` (`RUN_DEST`) does not exist, and the Drive registry must still have the SHA-256 of the pre-check (step 10). The notebook does not create `run/`: `claim` re-runs every pre-claim check and the registry rule (exactly the v1 pair, no v2 record; nothing is written if one fails), then appends the `started` record for the v2 model set to the live registry on Drive (`REGISTRY`), re-reads it, and only then creates `run/runner/` (`CLAIM_OUT`, with `run/`) by an exclusive `mkdir` and writes `claim.json`, `environment.json` and `pip_freeze.txt` into it. So a failure before the claim leaves no folder behind, and a permitted pre-claim rerun is not blocked by an empty `run/`. It reads no held-out audio (an `aasist` gate reads val-A). The SHA-256 of the three files is recorded right away (`CLAIM_HASHES`) and re-checked after the flush and remount.

The runner writes straight to Drive, not to `/content`, because the `scoring_started.json` marker that `score` creates in this folder must survive a runtime crash, so that the no-rerun rule holds in code and not only by procedure ([docs/stage11_run.md](../docs/stage11_run.md) Why (l)).

**Rerun rule (D46 (8) and D48 (7), pre-declared for the v2 claim): before the claim, an infrastructure failure (Drive, clone, pip, archive, artifact copies) may be rerun from the same COMMIT; an environment-gate failure (including require_versions), a reproduction-gate failure, a registry pre-check mismatch, or a rehearsal that ends with a failed check (failed_check not null) stops the v2 Stage 11 run until a logged decision; a rehearsal that crashes on an infrastructure error (failed_check null) may be rerun from the same COMMIT. After the claim, there is no rerun ever without a new decision, including a crash before any score exists; everything written stays for team review. The v2 claim cannot be repeated. Nothing in this notebook reads or reacts to the test results before they are exported.**

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
CLAIM_OUT = RUN_DEST / 'runner'
refuse_existing_outputs()
assert sha256(REGISTRY) == REGISTRY_PRE_CHECK['drive']['sha256'], 'the Drive registry changed since the pre-check'
assert_clean()
stream([sys.executable, '-m', RUNNER, 'claim', '--root', str(AUDIO_ROOT), *INPUTS, '--registry', str(REGISTRY),
        '--results-ref', RESULTS_REF, '--out', str(CLAIM_OUT)], cwd=REPO)
assert_clean()
CLAIM_FILES = ('claim.json', 'environment.json', 'pip_freeze.txt')
assert sorted(p.name for p in CLAIM_OUT.iterdir()) == sorted(CLAIM_FILES)
CLAIM_HASHES = {name: sha256(CLAIM_OUT / name) for name in CLAIM_FILES}
print('Claim written to', CLAIM_OUT)

## 14. Drive flush, remount and verify the claim (D46 (6))

On Colab, Drive writes go to a local cache first; reading the registry on the same mount would read that cache. Drive is therefore flushed, unmounted and remounted, and the claim is checked on what Drive actually stored: `claim.json` names `COMMIT`, `RESULTS_REF`, the winner and the bundle seed of step 5; the registry's SHA-256 must equal `registry_sha256` in `claim.json`; the registry holds the reviewed committed lines unchanged plus exactly one new line, and that last line is this run's `started` record with the v2 model set of `claim.json`; the three claim files in `run/runner/` must still match `CLAIM_HASHES`. `RUN_ID` is read from `claim.json` in code; it is never typed into this notebook. Any failure here happens after the claim: stop, keep everything, and report (D46 (8)).

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
CLAIM = json.loads((CLAIM_OUT / 'claim.json').read_text(encoding='utf-8'))
RUN_ID = CLAIM['run_id']
assert CLAIM['code_commit'] == COMMIT, CLAIM['code_commit']
assert CLAIM['results_ref'] == RESULTS_REF, CLAIM['results_ref']
assert CLAIM['architecture'] == WINNER and CLAIM['bundle_seed'] == BUNDLE_SEED, (CLAIM['architecture'], CLAIM['bundle_seed'])
assert len(CLAIM['model_ids']) == 3, CLAIM['model_ids']
assert sha256(REGISTRY) == CLAIM['registry_sha256'], 'the registry on Drive is not the one the claim recorded'
REGISTRY_BYTES = REGISTRY.read_bytes()
assert REGISTRY_BYTES.startswith(COMMITTED_REGISTRY.read_bytes()), 'the claim must only append to the reviewed registry'
assert len(REGISTRY_BYTES.splitlines()) == REGISTRY_PRE_CHECK['drive']['lines'] + 1, 'exactly one new registry line'
LAST = json.loads(REGISTRY_BYTES.decode('utf-8').splitlines()[-1])
assert LAST['run_id'] == RUN_ID and LAST['status'] == 'started', (LAST['run_id'], LAST['status'])
assert LAST['code_commit'] == COMMIT and LAST['split'] == CLAIM['split'], LAST
assert LAST['model_set'] == CLAIM['model_ids'], (LAST['model_set'], CLAIM['model_ids'])
assert {name: sha256(CLAIM_OUT / name) for name in CLAIM_FILES} == CLAIM_HASHES, 'claim files changed on Drive'
print('Claim verified on Drive after remount: run_id', RUN_ID, 'model set', CLAIM['model_ids'])

## 15. Score

`score` takes the claim's folder on Drive (`CLAIM_OUT`, `run/runner/`), `RUN_ID` from `claim.json`, the Drive registry and the same inputs as the claim, plus the FoR root. It checks `claim.json` against `RUN_ID` and the commit, creates the once-only marker `scoring_started.json`, records the scoring session's environment, re-runs every check (including the reproduction gate), checks that the registry holds the v1 pair and exactly this started v2 record, then computes the winner's inputs twice in memory (never cached, D46 (7)), writes `features.json` before any model sees them, scores the three seeds twice on CPU, writes `clip_scores/` and `run_report.json`, and records `complete` in the registry. Afterwards only identity is checked: `run_report.json` records `COMMIT` and `RUN_ID`, and the registry's last line is this run's `completed` record. Nothing is read from the scores.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
assert_clean()
stream([sys.executable, '-m', RUNNER, 'score', '--root', str(AUDIO_ROOT), *INPUTS, '--registry', str(REGISTRY),
        '--run-id', RUN_ID, '--out', str(CLAIM_OUT)], cwd=REPO)
assert_clean()
RUN_REPORT = json.loads((CLAIM_OUT / 'run_report.json').read_text(encoding='utf-8'))
assert RUN_REPORT['code_commit'] == COMMIT, RUN_REPORT['code_commit']
assert RUN_REPORT['run_id'] == RUN_ID, RUN_REPORT['run_id']
DONE = json.loads(REGISTRY.read_text(encoding='utf-8').splitlines()[-1])
assert DONE['run_id'] == RUN_ID and DONE['status'] == 'completed', (DONE['run_id'], DONE['status'])
assert DONE['results_ref'] == RESULTS_REF, DONE['results_ref']
print('Scored and completed: run_id', RUN_ID)

## 16. Report (PR V6a)

`src.evaluation.stage11_v2_report` reads only the stored run files (`run/runner/` on Drive), the registry, the F2 manifest and the committed v2 search and risk-v2 evidence (the policy is `docs/audit_evidence/v2_risk/risk_policy.json`, the file the runner used); it runs its integrity checks, computes the metrics under D57 as applied to the v2 model set and writes `metrics_report.json` and `metrics_report.md` into `/content/stage11_v2_report` (which must not exist). It is called once, exactly as [docs/stage11_report.md](../docs/stage11_report.md) ("v2 model set") defines it, and prints only the two paths. This notebook does not open, print or branch on any metric before step 18.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
REPORT_OUT = Path('/content/stage11_v2_report')
if REPORT_OUT.exists():
    raise FileExistsError(f'{REPORT_OUT} already exists')
assert_clean()
stream([sys.executable, '-m', 'src.evaluation.stage11_v2_report', '--run-dir', str(CLAIM_OUT),
        '--registry', str(REGISTRY), '--manifest', str(MANIFEST),
        '--search-evidence', str(SEARCH_EVIDENCE), '--risk-evidence', str(RISK_EVIDENCE),
        '--out', str(REPORT_OUT)], cwd=REPO)
assert_clean()
assert sorted(p.name for p in REPORT_OUT.iterdir()) == ['metrics_report.json', 'metrics_report.md']

## 17. Official export, flush, remount and re-verify

The runner's files are already on Drive in `run/runner/` and are not copied: the folder must hold exactly the three claim files, `scoring_started.json`, `environment_score.json`, `features.json`, `run_report.json` and `clip_scores/` with its 6 files (3 models × CSV + JSON sidecar, named after the model IDs in `claim.json`), the claim files must still match `CLAIM_HASHES`, and every file is hashed in place into the manifest. Copied to `run/` (`rehearsal/` created with `exist_ok=False`, no file overwritten) with `export_file`: `metrics_report.json` and `metrics_report.md`, and the rehearsal evidence under `rehearsal/`; each copy is re-hashed against its `/content` source. `stage11_v2_export_manifest.json` records notebook 06's fields without the CNN, baseline and MFCC ones, plus `search_version`, `risk_version`, the winner, its input kind and the bundle seed, the `require_versions` result, the registry pre-check, `run_id`, `results_ref`, and the registry's SHA-256 and line count after `complete` (harness "Run registry" (e)). Drive is then flushed, unmounted and remounted; every file under `run/` is hashed again against the manifest (a missing, extra or changed file fails) and the registry against its recorded SHA-256. These are candidate records until human review.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
SCORE_FILES = ('scoring_started.json', 'environment_score.json', 'features.json', 'run_report.json')
runner_files = sorted(p.relative_to(CLAIM_OUT).as_posix() for p in CLAIM_OUT.rglob('*') if p.is_file())
clip_files = sorted(p.name for p in (CLAIM_OUT / 'clip_scores').iterdir())
assert len(clip_files) == 6, clip_files
assert clip_files == sorted(f'{model_id}.{ext}' for model_id in CLAIM['model_ids'] for ext in ('csv', 'json')), clip_files
assert runner_files == sorted([*CLAIM_FILES, *SCORE_FILES, *(f'clip_scores/{name}' for name in clip_files)]), runner_files
assert {name: sha256(CLAIM_OUT / name) for name in CLAIM_FILES} == CLAIM_HASHES, 'claim files changed on Drive'
assert sorted(p.name for p in RUN_DEST.iterdir()) == ['runner'], sorted(p.name for p in RUN_DEST.iterdir())
(RUN_DEST / 'rehearsal').mkdir(exist_ok=False)
files = [drive_entry(CLAIM_OUT / rel) for rel in runner_files]
files += [export_file(REPORT_OUT / name, RUN_DEST / name) for name in ('metrics_report.json', 'metrics_report.md')]
files += [export_file(REHEARSAL_OUT / name, RUN_DEST / 'rehearsal' / name)
          for name in ('rehearsal_report.json', 'pip_freeze.txt')]
REGISTRY_SHA256 = sha256(REGISTRY)
REGISTRY_LINES = len(REGISTRY.read_text(encoding='utf-8').splitlines())
EXPORT_MANIFEST = {
    **base_manifest(),
    'run_id': RUN_ID,
    'results_ref': RESULTS_REF,
    'registry': {'path': REGISTRY.relative_to(DRIVE).as_posix(), 'sha256': REGISTRY_SHA256,
                 'lines': REGISTRY_LINES},
    'files': files,
}
MANIFEST_PATH = RUN_DEST / 'stage11_v2_export_manifest.json'
MANIFEST_TEXT = json.dumps(EXPORT_MANIFEST, indent=2, sort_keys=True) + '\n'
MANIFEST_PATH.write_text(MANIFEST_TEXT, encoding='utf-8')
print(MANIFEST_TEXT, end='')
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
checked = {'files': [*EXPORT_MANIFEST['files'], {
    'path': MANIFEST_PATH.relative_to(DRIVE).as_posix(),
    'sha256': hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest()}]}
print(f'Drive verified after remount: {verify_after_remount(RUN_DEST, checked)} files match the manifest.')
assert sha256(REGISTRY) == REGISTRY_SHA256, 'the registry changed after the export'
print('registry_sha256:', REGISTRY_SHA256, 'lines:', REGISTRY_LINES)

## 18. Results view

Only now, after the run is completed in the registry and every file is exported and verified on Drive, is the report shown. The numbers are reported whatever the outcome; nothing changes because of them: no threshold, model, policy or reporting convention is revised (D57, Protocol §12), and the deployed model is fixed by D48 (5), not by this result.

In [ ]:
assert MODE == 'official', 'dry run: stop before the claim'
from IPython.display import Markdown, display

display(Markdown((REPORT_OUT / 'metrics_report.md').read_text(encoding='utf-8')))

## 19. What is committed

After human review, a results PR commits:

- `docs/audit_evidence/run_registry.jsonl`: a byte copy of the Drive registry (the v1 pair plus the v2 `started` and `completed` records), with its SHA-256 and line count (D46 (6); they must equal `registry` in `stage11_v2_export_manifest.json`);
- under `docs/audit_evidence/stage11_v2_run/`: the seven runner JSON/TXT files (`claim.json`, `environment.json`, `pip_freeze.txt`, `scoring_started.json`, `environment_score.json`, `features.json`, `run_report.json`), `rehearsal_report.json`, `metrics_report.json` and `stage11_v2_export_manifest.json`;
- `metrics_report.md` at `RESULTS_REF` (`docs/results/stage11_v2_for_test.md`).

The per-clip CSVs and sidecars stay on Drive only (`stage11_v2/run/runner/clip_scores/`), bound by their SHA-256 in `run_report.json`, as risk-v2's `val_b_scores.npz` (AGENTS.md rule 7); Stage 14 reads them there (D48 (8)).

**Note for F5:** `src.bundle.f5` reads `docs/audit_evidence/stage11_v2_run/run_report.json` and `docs/audit_evidence/run_registry.jsonl` (a completed record of the v2 model set is required, D48 (7)), so the F5 build runs only after this results PR is merged.